# AGENTICAITA: Deliberative Multi-Agent Reasoning for Autonomous Trading
**ArXivist-generated reproduction notebook**  
Paper: [arxiv:2605.12532](https://arxiv.org/abs/2605.12532)  
Generated: 2026-05-16

Walks through core components interactively. No live exchange needed — all demos use synthetic data or DRY_RUN mode.

In [ ]:
import sys, importlib

print(f"Python: {sys.version}")
deps = ["aiohttp", "aiosqlite", "pydantic", "numpy", "yaml"]
for dep in deps:
    try:
        m = importlib.import_module(dep)
        ver = getattr(m, "__version__", "ok")
        print(f"  OK  {dep}: {ver}")
    except ImportError:
        print(f"  MISSING  {dep} -- run: pip install -r requirements.txt")
print("\nNote: asyncio system -- no GPU required.")

In [ ]:
import subprocess, sys, os
repo_root = os.path.abspath("..")
r = subprocess.run([sys.executable, "-m", "pip", "install", "-e", repo_root],
                   capture_output=True, text=True, cwd=repo_root)
print("OK" if r.returncode == 0 else r.stderr[-500:])

## Paper Overview

AGENTICAITA replaces signal-then-execute trading with an **autonomous deliberative loop**.

### Four contributions (Section 3):
1. **AZTE** — Adaptive Z-Score Trigger Engine: activate only on anomalous price moves
2. **SDP** — Sequential Deliberative Pipeline: Analyst -> Risk Manager -> Executor with typed JSON
3. **IGP** — Inference Gating Protocol: mutex preventing concurrent pipeline activations  
4. **CBD** — Correlation-Break Diversification: portfolio-aware asset prioritization

LLM backend: `qwen3.5:9b` via Ollama. No training. Proof-of-concept: 157 invocations, 76 assets, 5 days.

## Module 1: AZTE — Adaptive Z-Score Trigger Engine (Section 4.1)

$$r_t = \left|\frac{p_t - p_{t-1}}{p_{t-1}}\right| \quad \text{(Eq. 1)}$$

$$z_t = \frac{r_t - \hat{\mu}_W}{\hat{s}_W} \quad \text{(Eq. 2)}$$

$$\mathcal{T}_t = \mathbf{1}[z_t \geq 2.0] \vee \mathbf{1}[r_t \geq 0.003] \quad \text{(Eq. 3)}$$

In [ ]:
import sys; sys.path.insert(0, "..")
import numpy as np
from datetime import datetime, timedelta
from src.agenticaita.config import AZTEConfig
from src.agenticaita.azte import AdaptiveZScoreTriggerEngine

cfg = AZTEConfig(polling_interval_s=60, z_score_threshold=2.0,
                 rolling_window=30, absolute_return_floor=0.003)
azte = AdaptiveZScoreTriggerEngine(cfg)

# Simulate 40 price observations
np.random.seed(42)
prices = [60000.0]
for _ in range(39):
    prices.append(prices[-1] * (1 + np.random.normal(0, 0.001)))
prices[35] = prices[34] * 1.025  # inject 2.5% spike at t=35

ts = datetime(2026, 4, 6, 12, 0, 0)
for i, price in enumerate(prices):
    event = azte.update("BTC/USDT", price, ts + timedelta(minutes=i), per_asset_cooldown_s=0)
    if event:
        print(f"TRIGGER t={i}: z={event.z_score:.3f}  r={event.return_magnitude:.4f}  via {event.triggered_by}")

# Demonstrate equations directly
r_t = azte.compute_return_magnitude(prices[35], prices[34])
print(f"\nEq.1  r_t = {r_t:.4f}  (expected ~0.025)")

from collections import deque
dummy_returns = deque([abs(np.random.normal(0.001, 0.0005)) for _ in range(29)] + [r_t])
z = azte.compute_z_score(r_t, dummy_returns)
print(f"Eq.2  z_t = {z:.3f}  (above 2.0 => Eq.3 fires)")
triggered, by = azte.check_trigger(z, r_t)
print(f"Eq.3  T_t = {triggered}  via '{by}' ")

## Module 2: CBD — Correlation-Break Diversification (Section 4.5)

$$\rho^a_{cb} = 1 - \left|\rho\left(\{p^a\}, \{p^{BTC}\}\right)\right| \quad \text{(Eq. 9)}$$

$$\tilde{z}^a_t = \left(1 - e^{-\kappa(|z^a_t| - 2.0)}\right) \cdot \mathbf{1}[|z^a_t| \geq 2.0] \quad \text{(Eq. 10)}$$

$$\Omega^a = \alpha \tilde{z}^a_t + (1-\alpha)\rho^a_{cb}, \quad \alpha=0.5 \quad \text{(Eq. 11)}$$

In [ ]:
from src.agenticaita.config import CBDConfig
from src.agenticaita.cbd import CorrelationBreakDiversification
import numpy as np

cfg_cbd = CBDConfig(alpha=0.5, kappa=0.5)
cbd = CorrelationBreakDiversification(cfg_cbd)

np.random.seed(7)
btc = [60000 * (1 + np.random.normal(0, 0.01)) for _ in range(30)]
correlated   = [p * 0.95 + np.random.normal(0, 30) for p in btc]
decorrelated = [100 + np.random.normal(0, 2) for _ in range(30)]

for p_btc, p_c, p_d in zip(btc, correlated, decorrelated):
    cbd.update_btc_price(p_btc, window=30)
    cbd.update_price("CORR/USDT",  p_c, window=30)
    cbd.update_price("DECOR/USDT", p_d, window=30)

z_score = 2.8
z_tilde = cbd.compute_z_tilde(z_score)
print(f"Eq.10  z_tilde({z_score:.1f}) = {z_tilde:.4f}")

rho_c = cbd.compute_rho_cb("CORR/USDT",  30)
rho_d = cbd.compute_rho_cb("DECOR/USDT", 30)
print(f"Eq.9   rho_cb(CORR)  = {rho_c:.4f}  (low decorrelation)")
print(f"Eq.9   rho_cb(DECOR) = {rho_d:.4f}  (high decorrelation)")

omega_c = cbd.compute_omega(z_score, "CORR/USDT",  30)
omega_d = cbd.compute_omega(z_score, "DECOR/USDT", 30)
print(f"Eq.11  Omega(CORR)  = {omega_c:.4f}")
print(f"Eq.11  Omega(DECOR) = {omega_d:.4f}  <-- higher: incentivizes idiosyncratic alpha")

## Module 3: IGP — Inference Gating Protocol (Section 4.4)

**Definition 2**: Binary semaphore L. Admitted iff L=0; upon admission L←1; upon completion L←0.  
Concurrent triggers are **discarded** (not queued) and logged as `pipeline_busy`.

In [ ]:
import asyncio
from src.agenticaita.igp import InferenceGatingProtocol

async def demo_igp():
    igp = InferenceGatingProtocol(global_cooldown_s=0)

    ok1 = await igp.acquire("BTC/USDT")
    print(f"Trigger 1 acquired: {ok1}   (L=0 -> admitted)")

    ok2 = await igp.acquire("ETH/USDT")
    print(f"Trigger 2 acquired: {ok2}  (L=1 -> discarded as pipeline_busy)")

    await igp.release()
    ok3 = await igp.acquire("ETH/USDT")
    print(f"Trigger 3 (after release): {ok3}  (L=0 -> admitted)")
    await igp.release()

    print(f"Stats: {igp.get_stats()}")

asyncio.run(demo_igp())

## Module 4: Risk Manager Hard Gates (Section 4.2, Eq. 4–7)

Layer A — deterministic, no LLM call:
- **Eq. 4**: signal ∈ {long, short}
- **Eq. 5**: confidence ≥ 0.60  
- **Eq. 6**: |entry − stop_loss| / entry ≤ 0.02  
- **Eq. 7**: size_usd ≤ $500

In [ ]:
from src.agenticaita.config import RiskManagerConfig
from src.agenticaita.agents.risk_manager import RiskManagerAgent
from src.agenticaita.schemas import AnalystOutput

rm_cfg = RiskManagerConfig(confidence_gate=0.60, max_stop_loss_pct=0.02, max_position_usd=500)
rm = RiskManagerAgent(cfg=rm_cfg, llm=None)

cases = [
    ("long",  0.75, 60000, 58800, 250, "Valid  -- all gates pass"),
    ("wait",  0.75, 60000, 58800, 250, "FAIL Eq.4: signal=wait"),
    ("long",  0.45, 60000, 58800, 250, "FAIL Eq.5: confidence < 0.60"),
    ("long",  0.75, 60000, 56000, 250, "FAIL Eq.6: SL too wide (6.7%)"),
    ("long",  0.75, 60000, 58800, 750, "FAIL Eq.7: size > $500"),
]

for signal, conf, entry, sl, size, desc in cases:
    out = AnalystOutput(signal=signal, confidence=conf, entry_price=entry,
                       stop_loss=sl, take_profit=entry*1.03, size_usd=size, reasoning="test")
    r = rm.hard_gate_check(out)
    tag = "PASS" if r.passed else f"FAIL ({r.failed_gate})"
    print(f"  {tag:<28} {desc}")

## Transaction Cost Model (Section 4.6, Eq. 12–13)

$$C^{rt}_i = Q_i P_i f_{\text{taker}} + \frac{1}{2}Q_i|P_{\text{ask}} - P_{\text{bid}}| + \lambda \sigma_i \sqrt{\frac{Q_i}{V_i P_i}}$$

Where $\lambda=0.8$ is the market impact coefficient.

In [ ]:
import numpy as np

def round_trip_cost(Q, P, f_taker, spread, sigma, V, lam=0.8):
    '''Eq. 13: Round-trip transaction cost.
    Q: position size (contracts), P: price, f_taker: taker fee,
    spread: bid-ask spread, sigma: volatility, V: daily volume, lam: impact coef'''
    exchange_fee  = Q * P * f_taker
    half_spread   = 0.5 * Q * spread
    market_impact = lam * sigma * np.sqrt(Q / (V * P))
    return exchange_fee + half_spread + market_impact

# Example: BTC/USDT trade at $60,000
scenarios = [
    ("Zero cost (idealized)",  0.0,   0.0,    0.001, 1e6, 0.8),
    ("Conservative",           0.001, 60.0,   0.001, 1e6, 0.8),
    ("Realistic (Table 7)",    0.001, 120.0,  0.015, 5e5, 0.8),
    ("Adverse",                0.002, 300.0,  0.02,  1e5, 0.8),
]

Q, P = 0.004, 60000.0  # $240 notional
print(f"Trade: {Q} BTC @ ${P:,.0f}  (notional ${Q*P:.0f})")
print("-" * 55)
for name, f, spread, sigma, V, lam in scenarios:
    c = round_trip_cost(Q, P, f, spread, sigma, V, lam)
    pct = c / (Q * P) * 100
    print(f"  {name:<28} cost=${c:.4f} ({pct:.3f}%)")

## Agentic Friction Metric (Section 5.1, Eq. 8)

$$F = \frac{N_{\text{rej}} + N_{\text{wait}}}{N}$$

Paper reported $F = 11.5\%$: 13 Analyst self-abstentions + 5 RM rejections = 18/157.

In [ ]:
def agentic_friction(N_total, N_wait, N_reject):
    '''Eq. 8: Fraction of invocations resulting in non-execution.'''
    F = (N_wait + N_reject) / N_total
    return F

# Paper values (Table 5)
N = 157; N_wait = 13; N_rej = 5
F = agentic_friction(N, N_wait, N_rej)
print(f"Eq.8  F = ({N_wait} + {N_rej}) / {N} = {F:.4f} = {F*100:.1f}%")
print(f"Paper reported: 11.5%  -- match: {'YES' if abs(F - 0.115) < 0.001 else 'NO'}")

# Sensitivity analysis
print("\nSensitivity: agentic friction vs abstention rate")
for n_wait in [5, 10, 13, 20, 30]:
    f = agentic_friction(157, n_wait, 5)
    print(f"  N_wait={n_wait:2d}  F={f*100:.1f}%")

## Paper Results Summary (Table 5)

In [ ]:
results = {
    "Total invocations":          157,
    "Assets monitored":           117,
    "Assets with trades":         76,
    "Trades executed":            139,
    "Win rate":                   "51.80% (72/139)",
    "Agentic Friction F":         "11.5%",
    "  Analyst self-abstention":  "8.3% (13/157)",
    "  RM rejection":             "3.2% (5/157)",
    "Net PnL":                    "-$15.07 (-0.058%)",
    "Profit factor":              0.841,
    "Mean risk/reward":           "3.02:1",
    "Break-even WR at 3.02:1":   "24.9%",
    "Max drawdown":               "$32.30",
    "Alpha vs BTC buy-hold":      "+$3,896 (+14.94pp)",
    "Binomial p-value":           "~0.34 (NOT significant, n=139)",
}
print("Table 5 -- AGENTICAITA DRY_RUN Results (Apr 6-11 2026)")
print("-" * 55)
for k, v in results.items():
    print(f"  {k:<36} {v}")
print()
print("Note: n=139 insufficient for statistical inference.")
print("Authors recommend >= 500 trades / 90 days before drawing conclusions.")

## What to Do Next

### Run the full system
```bash
# 1. Start Ollama
ollama run qwen3.5:9b

# 2. Populate asset list
echo "BTC/USDT" > configs/assets.txt
echo "ETH/USDT" >> configs/assets.txt

# 3. DRY_RUN
python run.py --config configs/config.yaml --mode DRY_RUN

# 4. Inspect results
python inspect_trades.py --db data/episodic_memory.db
python compute_metrics.py --db data/episodic_memory.db
```

### Before going LIVE — resolve these stubs
| File | Issue | Confidence |
|------|-------|------------|
| `exchange.py` | Exchange identity unknown — implement real adapter | 0.50 |
| `market_data.py` | Set correct ccxt exchange ID | 0.60 |
| `ollama_client.py` | Temperature=0 assumed for determinism | 0.70 |

### Stage 6 — Results Comparator
After collecting live results, return to ArXivist with your metrics to compare against Table 5.